# 10. Formulación del problema
Antes de entrenar se fijan cuatro cosas: **variable objetivo**, **momento de predicción**, **métrica** y **baseline**.

In [1]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from flows.common.paths import SILVER, SILVER_NAME

# Los módulos con nombre numérico se importan con importlib (ver sección 3)
ml = importlib.import_module("flows.05_ml_models")

sns.set_theme(style="whitegrid", context="notebook")
X, y = ml.load_dataset()          # lee Silver; X con columnas candidatas, y = Survived
print(X.shape, y.mean())          # tasa de supervivencia ≈ 0.38

(891, 12) 0.3838383838383838


## Paso 40. Auditoría de variables y fuga de datos
Columnas disponibles en Silver:

In [2]:
pd.read_parquet(SILVER / SILVER_NAME).columns.tolist()

['PassengerId',
 'Survived',
 'Pclass',
 'Name',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare',
 'Cabin',
 'Embarked',
 'Person',
 'TCabin',
 'HasCabin',
 'Familia',
 'Alone',
 'Survivor',
 'TTitulo',
 'AgeOriginal',
 'AgeImputada',
 'NEmbarked',
 'NSex',
 'NCabin',
 'NTitulo']

## Paso 41. División estratificada entrenamiento / prueba
A partir de aquí **no se vuelve a tocar `X_test`** hasta la Parte 16.

In [3]:
X_train, X_test, y_train, y_test = ml.split(X, y)

print('Entrenamiento:', X_train.shape, f'tasa = {y_train.mean():.3f}')
print('Prueba:       ', X_test.shape, f'tasa = {y_test.mean():.3f}')

Entrenamiento: (712, 12) tasa = 0.383
Prueba:        (179, 12) tasa = 0.385


## Paso 42. Baselines: lo mínimo que un modelo debe superar
Se evalúan con validación cruzada estratificada **solo sobre el entrenamiento**.

In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, f1_score, make_scorer, precision_score,
                             recall_score)
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=ml.settings.cv_folds, shuffle=True,
                     random_state=ml.settings.random_state)

# zero_division=0: el modelo ingenuo nunca predice "sobrevive", así que precisión/F1 valen 0
scoring = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, zero_division=0),
    'recall': make_scorer(recall_score, zero_division=0),
    'f1': make_scorer(f1_score, zero_division=0),
}

# 1. Baseline ingenuo: siempre predice la clase más frecuente (no sobrevive)
res = cross_validate(DummyClassifier(strategy='most_frequent'), X_train, y_train,
                     cv=cv, scoring=scoring)
ingenuo = {m: res[f'test_{m}'].mean() for m in scoring}
ingenuo

{'accuracy': np.float64(0.6165763813651137),
 'precision': np.float64(0.0),
 'recall': np.float64(0.0),
 'f1': np.float64(0.0)}

In [5]:
# 2. Baseline "regla de género": predice 1 si Sex == 'female'
#    (no aprende nada, pero se evalúa en los mismos pliegues para comparar en igualdad)
filas = []
for _, idx_val in cv.split(X_train, y_train):
    X_val, y_val = X_train.iloc[idx_val], y_train.iloc[idx_val]
    pred = (X_val['Sex'] == 'female').astype(int)
    filas.append({
        'accuracy': accuracy_score(y_val, pred),
        'precision': precision_score(y_val, pred),
        'recall': recall_score(y_val, pred),
        'f1': f1_score(y_val, pred),
    })
genero = pd.DataFrame(filas)
genero.mean()

accuracy     0.789382
precision    0.744527
recall       0.688687
f1           0.714253
dtype: float64

In [6]:
# 3. Comparación (copia estos valores en reports/problema_ml.md como "criterio de éxito")
baselines = pd.DataFrame({'Ingenuo (most_frequent)': ingenuo,
                          'Regla de género': genero.mean()}).T.round(3)
baselines

,accuracy,precision,recall,f1
Ingenuo (most_frequent),0.617,0.000,0.000,0.000
Regla de género,0.789,0.745,0.689,0.714


# 11. Preprocesamiento reproducible con `Pipeline`
Un `Pipeline` encadena imputar → codificar → modelar en un solo objeto. Al hacer `fit`, cada paso aprende **solo** de los datos de entrenamiento; al predecir, aplica lo aprendido.

## Paso 43. Demostrar la fuga de datos

In [7]:
media_todos = X.loc[X['Sex'] == 'female', 'Age'].mean()
media_train = X_train.loc[X_train['Sex'] == 'female', 'Age'].mean()

print(f'Edad media de mujeres con TODOS los datos: {media_todos:.3f}')
print(f'Edad media de mujeres solo con X_train:    {media_train:.3f}')
print(f'Diferencia:                                {media_todos - media_train:+.3f} años')

Edad media de mujeres con TODOS los datos: 27.916
Edad media de mujeres solo con X_train:    28.183
Diferencia:                                -0.268 años


## Paso 44. Imputación de edad por grupo como transformador
`GroupAgeImputer` (en `flows/05_ml_models.py`) aprende los promedios en `fit` y los aplica en `transform`. Si aparece un grupo no visto usa el promedio global del entrenamiento.

In [8]:
imputador = ml.GroupAgeImputer(group_cols=('Sex', 'Pclass')).fit(X_train)
imputador.group_means_.round(2)

AttributeError: module 'flows.05_ml_models' has no attribute 'GroupAgeImputer'

## Paso 45. Pipeline completo: ¿imputar por `Sex` o por `Sex` y `Pclass`?
`build_pipeline` imputa por `Sex` y `Pclass`. Se compara con la variante del Paso 22 (solo `Sex`) usando validación cruzada con regresión logística.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

pipe_sex_clase = ml.build_pipeline(LogisticRegression(max_iter=1000))
pipe_solo_sex = ml.build_pipeline(LogisticRegression(max_iter=1000)).set_params(
    age_imputer__group_cols=('Sex',))

auc_sex_clase = cross_val_score(pipe_sex_clase, X_train, y_train, cv=cv, scoring='roc_auc')
auc_solo_sex = cross_val_score(pipe_solo_sex, X_train, y_train, cv=cv, scoring='roc_auc')

print(f'Imputar por Sex + Pclass: ROC-AUC = {auc_sex_clase.mean():.3f} ± {auc_sex_clase.std():.3f}')
print(f'Imputar solo por Sex:     ROC-AUC = {auc_solo_sex.mean():.3f} ± {auc_solo_sex.std():.3f}')

## Paso 46. Verificar el preprocesamiento

In [ ]:
from sklearn.dummy import DummyClassifier

pipe = ml.build_pipeline(DummyClassifier())
pre = pipe[:-1].fit(X_train)           # todo el pipeline menos el modelo
Z = pre.transform(X_test)

print('¿Quedan NaN?', np.isnan(Z).any())
print('Columnas después del preprocesamiento:', Z.shape[1])

In [ ]:
nombres = pre['prep'].get_feature_names_out()
list(nombres)

In [ ]:
# Sex quedó en dos columnas one-hot, sin orden implícito
[n for n in nombres if 'Sex' in n]

# 12. Modelos candidatos y validación cruzada
La validación cruzada estratificada de 5 pliegues da 5 puntajes por modelo: la **media** estima el desempeño y la **desviación** su estabilidad.

## Paso 47. Modelos candidatos

In [ ]:
candidatos = ml.get_candidates(ml.settings.random_state)
candidatos

## Paso 48. Validación cruzada de todos los candidatos

In [ ]:
tabla_cv, pliegues = ml.cross_validate_candidates(X_train, y_train, return_folds=True)

## Paso 49. Leer los resultados

In [ ]:
cols_media = [f'{m}_mean' for m in ml.SCORING] + ['roc_auc_std', 'train_roc_auc_mean', 'gap_roc_auc']
tabla_cv[cols_media].round(3)

In [ ]:
orden = tabla_cv.index.tolist()
sns.boxplot(data=pliegues, x='model', y='roc_auc', order=orden)
plt.title('ROC-AUC por pliegue (validación cruzada)')
plt.xlabel('Modelo')
plt.ylabel('ROC-AUC')
plt.show()

In [ ]:
# Brecha entrenamiento - validación: cuanto mayor, más sobreajuste
tabla_cv['gap_roc_auc'].sort_values(ascending=False).round(3)

## Paso 50. Elegir finalistas
Se descartan `dummy` (baseline) y los modelos con peor ROC-AUC o mayor brecha. Se eligen los 3 mejores por ROC-AUC medio.

In [ ]:
finalistas = [m for m in tabla_cv.index if m != 'dummy'][:3]
print('Finalistas:', finalistas)
tabla_cv.loc[finalistas, ['roc_auc_mean', 'roc_auc_std', 'gap_roc_auc', 'accuracy_mean', 'f1_mean']].round(3)

# 13. Evaluación en profundidad
Se estudian las **predicciones fuera de muestra** del entrenamiento: cada pasajero se predice con un modelo que no lo vio, sin tocar el conjunto de prueba.

## Paso 51. Predicciones fuera de muestra y matriz de confusión

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

# Probabilidades fuera de muestra de cada finalista
probas = {
    nombre: cross_val_predict(ml.build_pipeline(candidatos[nombre]), X_train, y_train,
                              cv=cv, method='predict_proba')[:, 1]
    for nombre in finalistas
}
mejor = finalistas[0]
proba = probas[mejor]
pred = (proba >= 0.5).astype(int)
print('Modelo analizado:', mejor)

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_train, pred, display_labels=['No sobrevive', 'Sobrevive'])
plt.title(f'Matriz de confusión ({mejor}, umbral 0.5)')
plt.show()

In [ ]:
print(classification_report(y_train, pred, target_names=['No sobrevive', 'Sobrevive']))

**Lectura para el seguro:**
- **Verdaderos negativos:** pasajeros que no sobrevivieron y el modelo los marcó en riesgo. Correcto.
- **Falsos positivos:** el modelo dijo "sobrevive" a quien no sobrevivió. Es el error más costoso: subestima el riesgo.
- **Falsos negativos:** el modelo dijo "no sobrevive" a quien sí sobrevivió. Sobrestima el riesgo: cuesta dinero, no vidas.
- **Verdaderos positivos:** sobrevivientes identificados correctamente.

## Paso 52. Curvas ROC y de precisión-recall

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for nombre, p in probas.items():
    RocCurveDisplay.from_predictions(y_train, p, name=nombre, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_train, p, name=nombre, ax=axes[1])
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1)
axes[0].set_title('Curva ROC')
axes[1].set_title('Curva precisión-recall')
plt.tight_layout()
plt.show()

La curva precisión-recall es más informativa cuando la clase positiva es minoritaria (aquí ≈ 38 %): la ROC puede verse bien aunque el modelo cometa muchos falsos positivos, mientras que la PR muestra directamente cuánta precisión se pierde al aumentar el recall.

## Paso 53. Calibración y elección del umbral

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, precision_recall_curve

prob_true, prob_pred = calibration_curve(y_train, proba, n_bins=8)
plt.plot(prob_pred, prob_true, marker='o', label=mejor)
plt.plot([0, 1], [0, 1], 'k--', label='Calibración perfecta')
plt.title('Curva de calibración')
plt.xlabel('Probabilidad predicha')
plt.ylabel('Proporción real')
plt.legend()
plt.show()

print(f'Brier score: {brier_score_loss(y_train, proba):.3f}  (0 = perfecto, 0.25 = azar)')

In [ ]:
precision, recall, umbrales = precision_recall_curve(y_train, proba)
f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
umbral_f1 = float(umbrales[f1.argmax()])

pred_f1 = (proba >= umbral_f1).astype(int)
print(f'Umbral que maximiza F1: {umbral_f1:.2f}  ->  F1 = {f1.max():.3f}')
print(f'Umbral 0.5:                     ->  F1 = {f1_score(y_train, pred):.3f}')
print(f'Precisión con 0.5: {precision_score(y_train, pred):.3f} | con {umbral_f1:.2f}: {precision_score(y_train, pred_f1):.3f}')

In [ ]:
# Decisión del umbral. Como el error más costoso es el falso positivo (decir "sobrevive"
# a quien no sobrevivió), se usa el MAYOR entre 0.5 y el umbral de F1: nunca se baja el
# umbral a costa de la precisión.
umbral = round(max(0.5, umbral_f1), 2)
print('THRESHOLD elegido:', umbral)

## Paso 54. Análisis de errores por subgrupos

In [ ]:
analisis = X_train.copy()
analisis['y'] = y_train
analisis['pred'] = pred
analisis['proba'] = proba
analisis['error'] = analisis['pred'] != analisis['y']
analisis['GrupoEdad'] = pd.cut(analisis['Age'], bins=[0, 16, 40, 60, 100],
                               labels=['0-15', '16-39', '40-59', '60+'])

def metricas_grupo(g):
    positivos = g['y'] == 1
    return pd.Series({
        'n': len(g),
        'exactitud': (g['pred'] == g['y']).mean(),
        'recall': (g.loc[positivos, 'pred'] == 1).mean() if positivos.any() else np.nan,
    })

for col in ['Sex', 'Pclass', 'GrupoEdad']:
    print(f'\n--- {col} ---')
    print(analisis.groupby(col, observed=True)[['y', 'pred']].apply(metricas_grupo).round(3))

In [ ]:
# 10 errores de alta confianza: el modelo estaba muy seguro y se equivocó
errores = analisis[analisis['error']].copy()
errores['confianza'] = (errores['proba'] - 0.5).abs()
errores.nlargest(10, 'confianza')[['Sex', 'Pclass', 'Age', 'TTitulo', 'Familia', 'Fare', 'y', 'proba']]

## Paso 55. Curva de aprendizaje

In [ ]:
from sklearn.model_selection import learning_curve

sizes, tr, va = learning_curve(ml.build_pipeline(candidatos[mejor]), X_train, y_train, cv=cv,
                               train_sizes=np.linspace(0.2, 1.0, 5), scoring='roc_auc')

for puntajes, etiqueta in [(tr, 'Entrenamiento'), (va, 'Validación')]:
    media, desv = puntajes.mean(axis=1), puntajes.std(axis=1)
    plt.plot(sizes, media, marker='o', label=etiqueta)
    plt.fill_between(sizes, media - desv, media + desv, alpha=0.2)
plt.title(f'Curva de aprendizaje ({mejor})')
plt.xlabel('Pasajeros de entrenamiento')
plt.ylabel('ROC-AUC')
plt.legend()
plt.show()

print(f'Brecha final entrenamiento - validación: {tr.mean(axis=1)[-1] - va.mean(axis=1)[-1]:.3f}')

# 14. Ajuste de hiperparámetros
La búsqueda se hace **dentro** de la validación cruzada del entrenamiento. El mejor puntaje de la búsqueda es optimista: por eso se comprueba después si la mejora es real.

## Paso 56. Espacio de búsqueda

In [ ]:
rejillas = ml.get_param_grids()
rejillas

## Paso 57. Búsqueda con validación cruzada
Se buscan hiperparámetros para los finalistas que tienen rejilla.

In [ ]:
from sklearn.model_selection import GridSearchCV

busquedas = {}
for nombre in finalistas:
    if nombre not in rejillas:
        print(f'{nombre}: sin rejilla definida, se mantiene con sus valores por defecto')
        continue
    gs = GridSearchCV(ml.build_pipeline(candidatos[nombre]), rejillas[nombre],
                      scoring=ml.PRIMARY_METRIC, cv=cv, n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)                      # SOLO entrenamiento
    busquedas[nombre] = gs
    print(f'{nombre}: mejor ROC-AUC = {gs.best_score_:.3f} con {gs.best_params_}')

In [ ]:
# Las 5 mejores combinaciones de cada búsqueda: ¿qué tan plana es la superficie?
for nombre, gs in busquedas.items():
    res = pd.DataFrame(gs.cv_results_).sort_values('rank_test_score')
    print(f'\n--- {nombre} ---')
    print(res[['params', 'mean_test_score', 'std_test_score']].head(5).to_string(index=False))

In [ ]:
# Modelo elegido: el de mayor ROC-AUC tras el ajuste (o el mejor finalista si ninguno tiene rejilla)
if busquedas:
    mejor = max(busquedas, key=lambda n: busquedas[n].best_score_)
    mejores_params = busquedas[mejor].best_params_
else:
    mejores_params = {}
print('Modelo elegido:', mejor)
print('Hiperparámetros:', mejores_params)

## Paso 58. ¿Mejoró de verdad? (comparación pareada)
Se usan los **mismos pliegues** para el modelo con valores por defecto y el ajustado, y se mira la diferencia pliegue a pliegue.

In [ ]:
auc_defecto = cross_val_score(ml.build_pipeline(candidatos[mejor]), X_train, y_train,
                              cv=cv, scoring='roc_auc')
auc_ajustado = cross_val_score(ml.build_pipeline(candidatos[mejor]).set_params(**mejores_params),
                               X_train, y_train, cv=cv, scoring='roc_auc')
diferencias = auc_ajustado - auc_defecto

print(f'Por defecto: {auc_defecto.mean():.3f} ± {auc_defecto.std():.3f}')
print(f'Ajustado:    {auc_ajustado.mean():.3f} ± {auc_ajustado.std():.3f}')
print(f'Diferencia por pliegue: media {diferencias.mean():+.3f}, desviación {diferencias.std():.3f}')
print('¿La mejora supera la variabilidad entre pliegues?', diferencias.mean() > diferencias.std())

# 15. Interpretación y ablación de variables
La importancia de una variable mide cuánto le sirve al modelo para **predecir**, no qué **causa** sobrevivir.

## Paso 59. Coeficientes del modelo lineal

In [ ]:
pipe_lr = ml.build_pipeline(LogisticRegression(max_iter=1000)).fit(X_train, y_train)
coefs = pd.DataFrame({
    'variable': pipe_lr['prep'].get_feature_names_out(),
    'coef': pipe_lr['model'].coef_[0],
})
coefs['odds_ratio'] = np.exp(coefs['coef'])
coefs.reindex(coefs['coef'].abs().sort_values(ascending=False).index).round(3)

## Paso 60. Importancia por permutación
Se baraja cada columna original en datos **no vistos** (una partición de validación dentro del entrenamiento) y se mide cuánto cae el ROC-AUC.

In [ ]:
from sklearn.inspection import permutation_importance

X_tr, X_val, y_tr, y_val = ml.split(X_train, y_train, test_size=0.25)    # NO se usa X_test
pipe_perm = ml.build_pipeline(candidatos[mejor]).set_params(**mejores_params).fit(X_tr, y_tr)

columnas_modelo = ml.NUM_FEATURES + ml.CAT_FEATURES
r = permutation_importance(pipe_perm, X_val[columnas_modelo], y_val,
                           scoring='roc_auc', n_repeats=30, random_state=42)
importancias = pd.DataFrame({'variable': columnas_modelo, 'media': r.importances_mean,
                             'desv': r.importances_std}).sort_values('media')

plt.barh(importancias['variable'], importancias['media'], xerr=importancias['desv'])
plt.title(f'Importancia por permutación ({mejor})')
plt.xlabel('Caída en ROC-AUC al barajar la variable')
plt.show()

## Paso 61. Ablación: ¿cuánto aporta cada grupo de variables?
Se reentrena con y sin cada grupo, con los **mismos pliegues**.

In [ ]:
base_num, base_cat = ml.NUM_FEATURES, ml.CAT_FEATURES
configuraciones = {
    'A) base': (base_num, base_cat),
    'B) base + cabina': (base_num + ml.CABIN_NUM, base_cat + ml.CABIN_CAT),
    'C) base sin TTitulo': (base_num, [c for c in base_cat if c != 'TTitulo']),
    'D) base sin Sex': (base_num, [c for c in base_cat if c != 'Sex']),
}
filas = []
for nombre, (num, cat) in configuraciones.items():
    pipe_ab = ml.build_pipeline(candidatos[mejor], num, cat).set_params(**mejores_params)
    s = cross_val_score(pipe_ab, X_train, y_train, cv=cv, scoring='roc_auc')
    filas.append({'configuracion': nombre, 'roc_auc_media': s.mean(), 'roc_auc_desv': s.std()})

ablacion = pd.DataFrame(filas)
ablacion['delta_vs_A'] = ablacion['roc_auc_media'] - ablacion.loc[0, 'roc_auc_media']
ablacion.round(3)

In [ ]:
# Decisión sobre las variables de cabina: se excluyen aunque mejoren un poco la métrica,
# porque la cabina pudo registrarse más para quienes sobrevivieron (artefacto del registro),
# y un modelo de riesgo no debe apoyarse en eso.
usar_cabina = False
print('USE_CABIN_FEATURES =', usar_cabina)

- **Cabina (B):** la columna `delta_vs_A` muestra cuánto cambia el ROC-AUC al añadirla. Aunque la mejora sea positiva, se excluye por el posible artefacto de registro discutido en el Paso 40: el modelo estaría aprendiendo quién quedó registrado, no quién estaba en riesgo.
- **Sin `TTitulo` (C) y sin `Sex` (D):** el desempeño cae poco porque cada una "toma el relevo" de la otra: el título codifica el sexo (Mr vs. Mrs/Miss).

# 16. Evaluación final sobre el conjunto de prueba
Todas las decisiones se tomaron con el entrenamiento. Ahora se entrena el modelo elegido con **todo** `X_train` y se evalúa **una sola vez** en `X_test`.

## Paso 62. Evaluar en prueba con incertidumbre

In [ ]:
num_final = ml.NUM_FEATURES + (ml.CABIN_NUM if usar_cabina else [])
cat_final = ml.CAT_FEATURES + (ml.CABIN_CAT if usar_cabina else [])

pipe_final = ml.build_pipeline(candidatos[mejor], num_final, cat_final).set_params(**mejores_params)
pipe_final.fit(X_train, y_train)

resultado_prueba = ml.evaluate_on_test(pipe_final, X_test, y_test, threshold=umbral)   # UNA vez
resultado_prueba

In [ ]:
proba_test = pipe_final.predict_proba(X_test)[:, 1]
ic_bajo, ic_alto = ml.bootstrap_ci(y_test, proba_test)
print(f"ROC-AUC en prueba: {resultado_prueba['roc_auc']:.3f}  (IC 95 %: {ic_bajo:.3f} - {ic_alto:.3f})")
print(f"Ancho del intervalo: {ic_alto - ic_bajo:.3f} con {resultado_prueba['n_test']} pasajeros de prueba")

proba_test = pipe_final.predict_proba(X_test)[:, 1]
ic_bajo, ic_alto = ml.bootstrap_ci(y_test, proba_test)
print(f"ROC-AUC en prueba: {resultado_prueba['roc_auc']:.3f}  (IC 95 %: {ic_bajo:.3f} - {ic_alto:.3f})")
print(f"Ancho del intervalo: {ic_alto - ic_bajo:.3f} con {resultado_prueba['n_test']} pasajeros de prueba")

## Paso 63. Comparar y documentar limitaciones

In [ ]:
pred_genero_test = (X_test['Sex'] == 'female').astype(int)
comparacion = pd.DataFrame({
    'Ingenuo': {'CV exactitud': ingenuo['accuracy'], 'CV F1': ingenuo['f1'],
                'Prueba exactitud': (y_test == 0).mean(), 'Prueba F1': 0.0},
    'Regla de género': {'CV exactitud': genero['accuracy'].mean(), 'CV F1': genero['f1'].mean(),
                        'Prueba exactitud': accuracy_score(y_test, pred_genero_test),
                        'Prueba F1': f1_score(y_test, pred_genero_test)},
    f'Modelo ({mejor})': {'CV exactitud': tabla_cv.loc[mejor, 'accuracy_mean'],
                          'CV F1': tabla_cv.loc[mejor, 'f1_mean'],
                          'Prueba exactitud': resultado_prueba['accuracy'],
                          'Prueba F1': resultado_prueba['f1']},
}).T.round(3)
comparacion

In [ ]:
print(f"ROC-AUC validación cruzada: {tabla_cv.loc[mejor, 'roc_auc_mean']:.3f} ± {tabla_cv.loc[mejor, 'roc_auc_std']:.3f}")
print(f"ROC-AUC prueba:             {resultado_prueba['roc_auc']:.3f}  (IC 95 %: {ic_bajo:.3f} - {ic_alto:.3f})")

In [ ]:
print(f'FINAL_ESTIMATOR = "{mejor}"')
print('FINAL_PARAMS = {')
for k, v in mejores_params.items():
    print(f'    "{k}": {v!r},')
print('}')
print(f'USE_CABIN_FEATURES = {usar_cabina}')
print(f'THRESHOLD = {umbral}')

# 17. Modelo final guardado y prueba de humo
El modelo definitivo lo entrena `make train` (o `make pipeline`) con las decisiones congeladas y **todos** los datos etiquetados, y lo guarda en `models/` con sus metadatos.

## Paso 66. Metadatos del modelo guardado

In [ ]:
from flows.common.paths import BRONZE, MODELS, BRONZE_UNLABELED_NAME

modelo_guardado, meta = ml.load_model(MODELS / 'titanic_model_v1.joblib')
{k: meta[k] for k in ['estimator', 'params', 'threshold', 'n_train', 'sklearn_version', 'saved_at']}

In [ ]:
meta['metrics']

## Paso 69. Prueba de humo con pasajeros sin etiqueta
Se comprueba que el flujo carga → preprocesamiento → predicción funciona sobre pasajeros que el modelo nunca vio y cuya respuesta no conocemos. No hay métrica que calcular.

In [ ]:
nuevos = pd.read_csv(BRONZE / BRONZE_UNLABELED_NAME)
X_nuevos = ml.features_from_bronze(nuevos)          # mismas transformaciones deterministas de Silver
predicciones = ml.predict(modelo_guardado, X_nuevos, threshold=meta['threshold'])

print('Pasajeros sin etiqueta:', len(X_nuevos))
print('¿Hay NaN en las predicciones?', predicciones.isna().any().any())
print(f"Proporción que el modelo predice como sobrevivientes: {predicciones['pred'].mean():.3f}")

In [ ]:
sns.histplot(predicciones['proba_survive'], bins=20)
plt.title('Probabilidad de supervivencia predicha (pasajeros sin etiqueta)')
plt.xlabel('Probabilidad predicha')
plt.ylabel('Pasajeros')
plt.show()

In [ ]:
# ¿Aparecen categorías que el modelo no vio en el entrenamiento?
for col in ml.CAT_FEATURES:
    nuevas = set(X_nuevos[col].dropna().unique()) - set(X[col].dropna().unique())
    print(f'{col}: {nuevas if nuevas else "sin categorías nuevas"}')
print('Valores faltantes en los datos nuevos:')
X_nuevos.isnull().sum()[lambda s: s > 0]